In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_7_try_0.pickle

In [3]:
%%RecordEvent
%%time
### cell 8 ###
def combine_subset_of_data_from_multiple_years_20(question_of_interest, x_axis_title, include_2017=None):
    # Define year-dataframe pairs in desired order
    pairs = [
        ('2018', responses_df_2018_cell10),
        ('2019', responses_df_2019_cell10),
        ('2020', responses_df_2020),
        ('2021', responses_df_2021),
        ('2022', responses_df_2022_cell10),
    ]
    if include_2017 is not None:
        pairs.insert(0, ('2017', responses_df_2017))
    # Precompute non-null counts per year
    denom = {year: df[question_of_interest].count() for year, df in pairs}
    # Concatenate all years into one DataFrame and rename column for x-axis
    combined = pd.concat(
        [df[[question_of_interest]].assign(year=year) for year, df in pairs],
        ignore_index=True
    ).rename(columns={question_of_interest: x_axis_title})
    # Compute counts (including NaNs) and then percentages
    counts = combined.groupby(['year', x_axis_title], dropna=False).size()
    percentages = (counts * 100).div(counts.index.get_level_values('year').map(denom)).round(1)
    # Build final DataFrame with desired column order
    return (
        percentages
        .reset_index(name='percentage')
        [['percentage', 'year', x_axis_title]]
    )

# Parameters and preprocessing
question_of_interest_cell20 = 'What is your age (# years)?'
title_for_x_axis_cell20 = ''
# Merge upper age buckets before computing
responses_df_2018_cell10[question_of_interest_cell20].replace(['70-79', '80+'], '70+', inplace=True)

# Combine and inspect
age_df_combined = combine_subset_of_data_from_multiple_years_20(
    question_of_interest_cell20,
    title_for_x_axis_cell20
)
age_df_combined.info()

/opt/conda/envs/elastic/lib/python3.10/site-packages/IPython/core/interactiveshell.py:2362: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  result = fn(*args, **kwargs)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55 entries, 0 to 54
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   percentage  55 non-null     float64
 1   year        55 non-null     object 
 2               55 non-null     object 
dtypes: float64(1), object(2)
memory usage: 1.4+ KB
CPU times: user 18.2 ms, sys: 0 ns, total: 18.2 ms
Wall time: 18 ms


In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_8_try_1.pickle